# nb_91_log_load_step

Tiny helper called by the Fabric pipeline after the Warehouse load. The load is a SQL script, not a notebook, so it cannot write to `ctl_pipeline_run` by itself. This notebook writes that one row, with the same failure classes as `nb_90`.

The pipeline calls it twice: once with `STATUS = "success"` and once, on the failure path, with `STATUS = "failed"` and the error text.

In [ ]:
# ---- Parameters (the pipeline sets these) ---------------------------------------------
PIPELINE_NAME = "maison-memory-daily"
PIPELINE_RUN_ID = ""
STATUS = "success"
ERROR_MESSAGE = ""
STEP_NAME = "warehouse_load"
STEP_ORDER = 7

In [ ]:
# ---- Failure classification -----------------------------------------------------------
# Order matters: our own check messages come first, then the platform, then bad input, then broken fit.
CLASS_RULES = [
    ("processing", ["reconciliation failed", "checks failed", "more than one row", "difference ", "reconcile"]),
    ("infrastructure", ["timed out", "timeout", "outofmemory", "out of memory", "capacity", "throttl", "too many requests",
                        "status code 429", "status code 503", "http 429", "http 503", "service unavailable", "connection reset",
                        "livy", "session expired", "executor lost", "tokenservice", "fetch notebook content"]),
    ("source", ["filenotfound", "file not found", "no such file", "path does not exist", "does not exist: ", "short_file",
                "missing_file", "corrupt", "malformed", "unable to infer schema", "cannot parse", "is not a valid csv"]),
    ("integration", ["table_or_view_not_found", "table or view not found", "cannot be found", "unresolved_column", "cannot resolve",
                     "ambiguous_reference", "mismatch", "incompatible", "schema", "datatype", "cannot cast", "no such table"]),
]
SUGGESTED_ACTION = {
    "source": "Check the landing files for the batch (ctl_file_log). Ask the source team for a corrected or complete file. Do not retry until the input changes.",
    "integration": "A table, column or type does not match what this step expects. Check that the earlier steps finished and that no schema changed.",
    "processing": "One of our own checks failed. Read the check output of this notebook (reconciliation or integrity cell) before changing anything.",
    "infrastructure": "The platform failed, not the data. Retry once. If it repeats, check capacity and the Spark session.",
    "unknown": "No rule matched. Read the full error text and classify it by hand, then add a rule here.",
}
RETRYABLE = {"infrastructure": 1, "source": 0, "integration": 0, "processing": 0, "unknown": 0}


def classify_failure(error_text):
    """Return (failure_class, retryable 0/1, suggested_action) for an error message."""
    t = (error_text or "").lower()
    for cls, words in CLASS_RULES:
        if any(w in t for w in words):
            return cls, RETRYABLE[cls], SUGGESTED_ACTION[cls]
    return "unknown", RETRYABLE["unknown"], SUGGESTED_ACTION["unknown"]


print("failure classes:", [c for c, _ in CLASS_RULES] + ["unknown"])

In [ ]:
# ---- Write the row --------------------------------------------------------------------
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType, TimestampType)

if not PIPELINE_RUN_ID:
    PIPELINE_RUN_ID = "manual-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
now = datetime.now(timezone.utc).replace(tzinfo=None)
failed = STATUS != "success"
cls, retry, action = classify_failure(ERROR_MESSAGE) if failed else (None, None, None)
schema = StructType([
    StructField("pipeline_run_id", StringType()), StructField("pipeline_name", StringType()),
    StructField("step_order", IntegerType()), StructField("step", StringType()), StructField("status", StringType()),
    StructField("started_at", TimestampType()), StructField("finished_at", TimestampType()), StructField("duration_s", LongType()),
    StructField("failure_class", StringType()), StructField("retryable", IntegerType()),
    StructField("error_message", StringType()), StructField("suggested_action", StringType()),
    StructField("is_simulated", IntegerType())])
row = Row(pipeline_run_id=PIPELINE_RUN_ID, pipeline_name=PIPELINE_NAME, step_order=STEP_ORDER, step=STEP_NAME,
          status="failed" if failed else "success", started_at=now, finished_at=now, duration_s=0, failure_class=cls,
          retryable=retry, error_message=(ERROR_MESSAGE or "")[:1500] or None, suggested_action=action, is_simulated=0)
spark.createDataFrame([row], schema=schema).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("ctl_pipeline_run")
print("logged:", STEP_NAME, row.status, cls or "")

In [ ]:
spark.table("ctl_pipeline_run").filter("pipeline_run_id = '3d41f06b-dd42-4eb3-884f-4435fb676965' and step in ('nb_21_silver_sales','_pipeline')").select("step","is_simulated","failure_class","error_message").show(truncate=120)